# [LAB 10] 지도학습 > 추천시스템 > KNN Basic
## #01. KNN Basic 알고리즘 개요
- 이웃을 (K-Nearest Neighbors) 를 기반으로 하는 전통적인 협업 필터링 알고리즘
- 비슷한 사용자 또는 비슷한 아이템을 찾아서 그들의 평점을 평균하여 예측
- Baseline 이 편향 보정 모델이라면 KNN Basic 은 유사도 기반 모델


### [1] KNN Basic 의 기본 개념
- 두가지 방식 존재

|구분|설명|
|-|-|
|User-based CF|나와 비슷한 사용자들이 준 평점을 참고|
|Item-based CF|내가 높게 준 아이템과 비슷한 아이템 참고|


### [2] 예측 공식 (User based)

| 기호         | 의미                       |
| ---------- | ------------------------ |
| (N_k(u))   | 사용자 u와 가장 유사한 k명 (이웃 집합) |
| (sim(u,v)) | 사용자 u와 v의 유사도            |
| (r_{vi})   | 이웃 사용자 v가 아이템 i에 준 평점    |



### [3] 유사도 종류

| 방법      | 설명           |
| ------- | ------------ |
| Cosine  | 벡터 각도 기반     |
| Pearson | 평균 제거 후 상관계수 |
| MSD     | 평균 제곱 차이 기반  |


### [4] 해석 예시
- 사용자 A, 영화 X 를 아직 보지 않았을 때
- A 와 비슷한 사용자 3명 존재
- 각각 영화 X 에 대한 평점이 4,5,4
- DBTKEH RKWND VUDRBS RUFRHK 4.3
- A 의 예측 평점은 4.3
 > 즉 나와 비슷한 사람의 선택을 따른 결과임



 ### [5] KNN Basic 의 의미
 - 협업 필터링의 대표적인 알고리즘
 - 직관적인 이해 가능
 - 설명 가능성 높음
 - 데이터가 희소하면 성능 저하 가능
 - 편향 보정 기능은 기본적으로 없음


### [6] BaselineOnly VS KNNBasic 비교

| 구분       | BaselineOnly | KNNBasic  |
| -------- | ------------ | --------- |
| 핵심 개념    | 편향 보정        | 유사도 기반    |
| 공식 구조    | μ + bᵤ + bᵢ  | 유사도 가중 평균 |
| 개인 성향 반영 | O            | 간접적       |
| 이웃 정보 활용 | X            | O         |
| 복잡도      | 낮음           | 중간        |
| 설명 가능성   | 중간           | 높음        |



#### 정리
- Baseline → “기본 성향 보정”
- KNNBasic → “비슷한 사람(또는 아이템)을 참고”

추천 시스템 이론에서
Baseline은 기준선 모델,
KNN은 본격적인 협업필터링의 시작 단계.

## #02.준비작업
### [1] 패키지 참조

In [30]:
from hossam import *
from pandas import DataFrame,merge

from surprise import Dataset , Reader , KNNBasic , accuracy
from surprise.model_selection import train_test_split,GridSearchCV,RandomizedSearchCV

### [2] 데이터셋 가져오기
#### 분석대상 - 평점 데이터

In [31]:
origin = load_data('ml100k-ratings')
origin.head()

943명의 사용자가 1,682편의 영화에 대해 남긴 100,000개의 평점 기록으로 구성된 명시적 평가 기반 추천 시스템 학습용 데이터셋 (출처: University of Minnesota)

컬럼명     의미
---------  ---------
user_id    사용자 ID
item_id    아이템 ID
rating     평점
timestamp  평가 시각



,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


#### 분석결과 맵핑 데이터 - 영화 정보

In [32]:
metadata = load_data('ml100k-metadata')
metadata.head()

ml100k-ratings에 포함된 영화 제목, 공개시기, 장르 정보를 담고 있는 데이터 (출처: University of Minnesota)


,item_id,title,release_date,IMDb_URL,unknown,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-1995,http://us.imdb.com/M/title-exact?Toy%20Story%20(1995),0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,http://us.imdb.com/M/title-exact?GoldenEye%20(1995),0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,http://us.imdb.com/M/title-exact?Four%20Rooms%20(1995),0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,http://us.imdb.com/M/title-exact?Get%20Shorty%20(1995),0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0


## #03. KNNBasic 모델 적합
### [1] DataFrame 을 Dataset 객체로 변환

scikit-surprise 는 일반 DataFrame 을 직접 학습하지 않고, 반드시 Reader 를 통해 변환된 Dataset 객체만 학습 가능함


#### 평점의 범위 확인

In [33]:
rating_min = origin['rating'].min()
rating_max = origin['rating'].max()
print(f'Rating 범위 : {rating_min} ~ {rating_max}')


Rating 범위 : 1 ~ 5


#### surprise 라이브러리에서 사용할 수 있도록 데이터셋을 변환

In [34]:
# 평점의 범위를 지정하여 Reader 객체 생성
reader = Reader (rating_scale = (rating_min,rating_max))

# Dataset 객체 생성 - 사용자 식별자 , 아이템 식별자, 평점만으로 구성된 데이터 구조 필요
data = Dataset.load_from_df(origin[['user_id','item_id','rating']],reader)


data

### [2] KNNBasic 모델의 주요 하이퍼 파라미터

필수 튜닝 대상: k, sim_options['name'], user_based


| 파라미터명                        | 핵심도 | 설명                               | 기본값   | GridSearch 권장값               |
| ---------------------------- | --- | -------------------------------- | ----- | ---------------------------- |
| `k`                          | ⭐⭐⭐ | 최근접 이웃 수 (참고할 사용자/아이템 개수)        | 40    | [20, 30, 40, 50, 60, 80]     |
| `min_k`                      | ⭐⭐  | 예측에 필요한 최소 이웃 수                  | 1     | [1, 3, 5, 10]                |
| `sim_options['name']`        | ⭐⭐⭐ | 유사도 계산 방식 (msd, cosine, pearson) | 'msd' | ['msd', 'cosine', 'pearson'] |
| `sim_options['user_based']`  | ⭐⭐⭐ | 사용자기반(True) / 아이템기반(False)       | True  | [True, False]                |
| `sim_options['min_support']` | ⭐⭐  | 유사도 계산 시 최소 공통 평가 수              | 1     | [1, 3, 5]                    |
| `verbose`                    | ✅   | 학습 로그 출력 여부                      | False | [True, False]                |


In [35]:
from surprise import KNNBasic
from surprise.model_selection import RandomizedSearchCV


# 하이퍼파라미터 그리드
param_grid = {
    "k": [30, 50],
    "min_k": [1, 3],
    "sim_options": {
        "name": ["msd", "cosine", "pearson"],
        "user_based": [True, False],
        "min_support": [1, 3, 5]
    }
}

# RandomizedSearchCV 객체 생성
gs = RandomizedSearchCV(
    KNNBasic,
    param_grid,
    measures=["rmse", "mae"],
    cv=5,
    n_jobs=-1,
    random_state=52  # 재현성 확보
)

# 학습
gs.fit(data)

# 최적 결과 출력
print("Best RMSE Score:", gs.best_score["rmse"])
print("Best Parameters:", gs.best_params["rmse"])


Best RMSE Score: 0.974134583102478
Best Parameters: {'k': 30, 'min_k': 3, 'sim_options': {'name': 'msd', 'user_based': True, 'min_support': 5}}


## #04.성능평가

### [1] 훈련,검증 데이터 분리

In [36]:
#데이터를 학습용과 테스트용으로 분할 (80% 학습 ,20% 테스트)
train_data , test_data = train_test_split(data,test_size = 0.2, random_state=52)


# 학습용과 테스트용 데이터의 크기 출력
print(f'Trainset 크기: {train_data.n_ratings} 개')
print(f'Testset 크기: {len(test_data)} 개')

Trainset 크기: 80000 개
Testset 크기: 20000 개


### [2] 최적 모델 재학습

In [37]:
# 최적 파라미터 추출
best_params  = gs.best_params['rmse']



# 모델 생성
best_model = KNNBasic(**best_params)


# 전체 데이터 학습
best_model.fit(train_data)



Computing the msd similarity matrix...
Done computing similarity matrix.


### [3] 예측값 생성

In [38]:
predictions = best_model.test(test_data)
predictions[:5] #예측 결과의 일부를 출력

[Prediction(uid=303, iid=679, r_ui=2.0, est=2.961052611202615, details={'actual_k': 30, 'was_impossible': False}),
 Prediction(uid=308, iid=163, r_ui=4.0, est=3.6921297070081267, details={'actual_k': 30, 'was_impossible': False}),
 Prediction(uid=327, iid=663, r_ui=4.0, est=3.7116288447244896, details={'actual_k': 30, 'was_impossible': False}),
 Prediction(uid=912, iid=479, r_ui=4.0, est=4.125460350517006, details={'actual_k': 30, 'was_impossible': False}),
 Prediction(uid=224, iid=329, r_ui=3.0, est=3.153031573095475, details={'actual_k': 30, 'was_impossible': False})]

### [4] 성능 평가 지표 생성

In [39]:
cv_rmse = gs.best_score['rmse']

# Train 예측 (trainset 전체를 test 형식으로 변환)
train_predictions = best_model.test(train_data.build_testset())

# Test 예측
test_predictions = best_model.test(test_data)

# 성능 계산
train_rmse = accuracy.rmse(train_predictions, verbose=False)
train_mae = accuracy.mae(train_predictions, verbose=False)

test_rmse = accuracy.rmse(test_predictions, verbose=False)
test_mae = accuracy.mae(test_predictions, verbose=False)


# 일반화 오차 차이
rmse_gap = test_rmse - train_rmse
mae_gap = test_mae - train_mae


# 과적합 판정 기준 (RMSE 기준)
# 기준: test RMSE가 train RMSE보다 0.05 이상 크면 과적합 의심
if rmse_gap > 0.05:
    overfit_flag = "과적합 의심"
else:
    overfit_flag = "정상"


# 성능평가표 생성
import pandas as pd

result_df = pd.DataFrame({
    "Model": ["BaselineOnly"],
    "Train_RMSE": [train_rmse],
    "Test_RMSE": [test_rmse],
    "RMSE_Gap": [rmse_gap],
    "Train_MAE": [train_mae],
    "Test_MAE": [test_mae],
    "MAE_Gap": [mae_gap],
    "Overfitting": [overfit_flag]
})

result_df


,Model,Train_RMSE,Test_RMSE,RMSE_Gap,Train_MAE,Test_MAE,MAE_Gap,Overfitting
0,BaselineOnly,0.756,0.977,0.222,0.591,0.771,0.180,과적합 의심


## #05.TopN 추천

### [1] 아직 평가하지 않은 아이템에 대한 예측 수행

#### 에측 결과 생성

In [40]:
# 아직 평가하지 않은 (user, item) 조합 생성
anti_testset = train_data.build_anti_testset()

# 예측 수행
predictions = best_model.test(anti_testset)

# 예측 결과 일부 확인
predictions[:5]


[Prediction(uid=234, iid=205, r_ui=3.5317375, est=4.139000452121666, details={'actual_k': 30, 'was_impossible': False}),
 Prediction(uid=234, iid=504, r_ui=3.5317375, est=3.7569486475889007, details={'actual_k': 30, 'was_impossible': False}),
 Prediction(uid=234, iid=73, r_ui=3.5317375, est=3.250778140875676, details={'actual_k': 30, 'was_impossible': False}),
 Prediction(uid=234, iid=475, r_ui=3.5317375, est=3.6258093141506746, details={'actual_k': 30, 'was_impossible': False}),
 Prediction(uid=234, iid=294, r_ui=3.5317375, est=3.0112659725514788, details={'actual_k': 30, 'was_impossible': False})]

#### 예측 결과 데이터 프레임 구성

In [41]:
import pandas as pd

pred_df = pd.DataFrame(
    predictions,
    columns=["user_id", "item_id", "true_rating", "pred_rating", "details"]
)

pred_df.head()


,user_id,item_id,true_rating,pred_rating,details
0,234,205,3.532,4.139,"{'actual_k': 30, 'was_impossible': False}"
1,234,504,3.532,3.757,"{'actual_k': 30, 'was_impossible': False}"
2,234,73,3.532,3.251,"{'actual_k': 30, 'was_impossible': False}"
3,234,475,3.532,3.626,"{'actual_k': 30, 'was_impossible': False}"
4,234,294,3.532,3.011,"{'actual_k': 30, 'was_impossible': False}"


### [3] 특정 사용자에 대한 상위 10개의 추천 영화 검색
#### 35번 사용자에 대한 TOP 10 추천 데이터

In [42]:
N = 10
user_id = 35

topn_df = pred_df[pred_df["user_id"] == user_id]

topn_df = (
    topn_df[["user_id", "item_id", "pred_rating"]]
    .sort_values(by="pred_rating", ascending=False)
    .head(N)
    .reset_index(drop=True)
)

topn_df


,user_id,item_id,pred_rating
0,35,272,4.646
1,35,22,4.640
2,35,603,4.636
3,35,64,4.574
4,35,50,4.560
5,35,174,4.531
6,35,318,4.519
7,35,313,4.511
8,35,98,4.493
9,35,496,4.465


#### 메타 데이터와 병합하여 영화 정보 생성

In [43]:
movie_df = topn_df.merge(metadata, on="item_id", how="left")
movie_df


,user_id,item_id,pred_rating,title,release_date,IMDb_URL,unknown,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,35,272,4.646,Good Will Hunting (1997),01-Jan-1997,http://us.imdb.com/M/title-exact?imdb-title-119217,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,35,22,4.640,Braveheart (1995),16-Feb-1996,http://us.imdb.com/M/title-exact?Braveheart%20(1995),0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0
2,35,603,4.636,Rear Window (1954),01-Jan-1954,http://us.imdb.com/M/title-exact?Rear%20Window%20(1954),0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0
3,35,64,4.574,"Shawshank Redemption, The (1994)",01-Jan-1994,"http://us.imdb.com/M/title-exact?Shawshank%20Redemption,%20The%20(1994)",0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
4,35,50,4.560,Star Wars (1977),01-Jan-1977,http://us.imdb.com/M/title-exact?Star%20Wars%20(1977),0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0
5,35,174,4.531,Raiders of the Lost Ark (1981),01-Jan-1981,http://us.imdb.com/M/title-exact?Raiders%20of%20the%20Lost%20Ark%20(1981),0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6,35,318,4.519,Schindler's List (1993),01-Jan-1993,http://us.imdb.com/M/title-exact?Schindler's%20List%20(1993),0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0
7,35,313,4.511,Titanic (1997),01-Jan-1997,http://us.imdb.com/M/title-exact?imdb-title-120338,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0
8,35,98,4.493,"Silence of the Lambs, The (1991)",01-Jan-1991,"http://us.imdb.com/M/title-exact?Silence%20of%20the%20Lambs,%20The%20(1991)",0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0
9,35,496,4.465,It's a Wonderful Life (1946),01-Jan-1946,http://us.imdb.com/M/title-exact?It's%20a%20Wonderful%20Life%20(1946),0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
